# PerioVision AI: train the bone-loss landmark model (CEJ, root apex, bone crest)

**How to use:** `Runtime` → `Change runtime type` → **T4 GPU** → Save, then `Runtime` → **Run all** and allow Google Drive.
It takes roughly 1-3 hours. If Colab disconnects, just Run all again: it resumes from Drive.

**Data:** [DenPAR](https://zenodo.org/records/16645076): 1000 periapical radiographs whose CEJ points, apex points and
alveolar bone-level lines were annotated in Labelbox and verified by dental specialists
([Scientific Data, 2025](https://www.nature.com/articles/s41597-025-05906-9)). Licence **CC BY 4.0**. It is downloaded
inside Colab; the official Training / Validation / Testing split is kept.

**What you get in `MyDrive/PerioVision/export/`:**
- `dental_landmark_yolov8n-pose.pt`: the trained landmark model (file name the app expects)
- `landmark_test_metrics.json`: honest test-set results, including bone-loss error in percentage points and stage agreement
- `conformal_calibration.json`: uncertainty calibration from the validation split, with its coverage measured on the test split

Then on your laptop: `.\run.ps1 install-models -From <downloaded export folder>`.

In [ ]:
# 0. Settings
MODEL   = "yolo11m-pose"
IMGSZ   = 1024
EPOCHS  = 200
PATIENCE = 40
BATCH   = -1
SEED    = 42
MIN_KPT_CONF = 0.3      # same threshold the app uses (backend/config/thresholds.json)

In [ ]:
# 1. GPU + Google Drive
import subprocess, os
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True).stdout or "NO GPU: Runtime > Change runtime type > T4 GPU")
from google.colab import drive
drive.mount("/content/drive")
ROOT = "/content/drive/MyDrive/PerioVision"
for d in ("runs", "export"):
    os.makedirs(f"{ROOT}/{d}", exist_ok=True)

In [ ]:
# 2. Same Ultralytics version as the PerioVision backend
!pip -q install "ultralytics==8.4.21"
import ultralytics; ultralytics.checks()

In [ ]:
# 3. Download DenPAR from Zenodo (141 MB) and check its checksum
import hashlib, zipfile, urllib.request
Z = "/content/denpar.zip"
if not os.path.isdir("/content/denpar/Dataset"):
    urllib.request.urlretrieve("https://zenodo.org/api/records/16645076/files/DenPAR%20Radiographs%20Dataset.zip/content", Z)
    md5 = hashlib.md5(open(Z, "rb").read()).hexdigest()
    assert md5 == "4fcecde6bfa4dc47daec83b1f7856c7f", f"checksum mismatch: {md5}"
    zipfile.ZipFile(Z).extractall("/content/denpar")
print(os.listdir("/content/denpar/Dataset"))

In [ ]:
%%writefile /content/convert_denpar.py
"""Convert the DenPAR periapical dataset to the YOLO-pose format PerioVision trains on.

DenPAR (Zenodo record 16645076, CC BY 4.0): 1000 periapical radiographs with, per image,
tooth boxes, tooth-wise masks, CEJ points, apex points and alveolar bone-level polylines,
annotated in Labelbox and verified by dental specialists. Points are NOT grouped per tooth,
so this script assigns them:

* each CEJ / apex point goes to the tooth whose mask is nearest (within MAX_DIST px);
* a tooth's apex = mean of its apex points (multi-rooted teeth have several);
* for each CEJ site (mesial / distal) the bone level = the deepest bone-line vertex that
  touches the tooth surface on that side, i.e. where bone meets the root;
* the tooth's keypoints come from its WORST site (largest bone loss), which is what
  periodontal staging uses.

Output per tooth line: 0 cx cy w h  cej_x cej_y v  apex_x apex_y v  crest_x crest_y v
(normalised; v = 2 visible, 0 missing). The official Training / Validation / Testing split is kept.

Usage (from backend/):
    python scripts/convert_denpar.py --src <folder containing Dataset/> --out <output folder>
"""
import argparse
import glob
import json
import os
import shutil

import cv2
import numpy as np

MAX_DIST = 25.0      # px: how far a point may lie from a tooth mask and still belong to it
SPLITS = {"Training": "train", "Validation": "val", "Testing": "test"}


def load_masks(folder):
    masks = []
    for f in sorted(glob.glob(os.path.join(folder, "*.png")), key=lambda p: int("".join(c for c in os.path.basename(p) if c.isdigit()) or 0)):
        m = cv2.imread(f, cv2.IMREAD_GRAYSCALE)
        if m is not None and m.max() > 0:
            masks.append(m > 127)
    return masks


def dist_map(mask):
    # distance (px) from every pixel to the nearest mask pixel
    return cv2.distanceTransform((~mask).astype(np.uint8), cv2.DIST_L2, 5)


def at(dm, p):
    h, w = dm.shape
    x, y = int(round(min(max(p[0], 0), w - 1))), int(round(min(max(p[1], 0), h - 1)))
    return float(dm[y, x])


def convert_image(kp, bone, masks):
    teeth = []
    dms = [dist_map(m) for m in masks]
    for m in masks:
        ys, xs = np.nonzero(m)
        teeth.append({"box": [xs.min(), ys.min(), xs.max(), ys.max()], "cej": [], "apex": []})

    def nearest(p):
        d = [at(dm, p) for dm in dms]
        i = int(np.argmin(d)) if d else -1
        return i if i >= 0 and d[i] <= MAX_DIST else -1

    for name, key in (("CEJ_Points", "cej"), ("Apex_Points", "apex")):
        for p in kp.get(name, []):
            i = nearest(p)
            if i >= 0:
                teeth[i][key].append(np.asarray(p, float))
    vertices = [np.asarray(v, float) for line in bone.get("Bone_Lines", []) for v in line]

    rows = []
    for i, t in enumerate(teeth):
        if not t["cej"] or not t["apex"]:
            continue                      # cannot define the root axis: skip the tooth entirely
        apex = np.mean(t["apex"], axis=0)
        cx_mid = np.mean([c[0] for c in t["cej"]])
        best = None
        for cej in t["cej"]:
            axis = apex - cej
            L2 = float(axis @ axis)
            if L2 < 1:
                continue
            side = np.sign(cej[0] - cx_mid) if len(t["cej"]) > 1 else 0
            cands = [v for v in vertices if at(dms[i], v) <= MAX_DIST
                     and (side == 0 or np.sign(v[0] - cx_mid) == side)]
            crest = max(cands, key=lambda v: float((v - cej) @ axis)) if cands else None
            ratio = float((crest - cej) @ axis) / L2 if crest is not None else -1
            if best is None or ratio > best[0]:
                best = (ratio, cej, crest)
        if best is None:
            continue
        rows.append({"box": t["box"], "cej": best[1], "apex": apex, "crest": best[2]})
    return rows


def main() -> int:
    ap = argparse.ArgumentParser()
    ap.add_argument("--src", required=True, help="folder that contains Dataset/")
    ap.add_argument("--out", required=True)
    args = ap.parse_args()
    root = os.path.join(args.src, "Dataset")
    stats = {}
    for split, short in SPLITS.items():
        os.makedirs(os.path.join(args.out, "images", short), exist_ok=True)
        os.makedirs(os.path.join(args.out, "labels", short), exist_ok=True)
        n_img = n_teeth = n_crest = 0
        for kp_path in sorted(glob.glob(os.path.join(root, split, "Key Points Annotations", "*.json"))):
            stem = os.path.splitext(os.path.basename(kp_path))[0]
            img_path = os.path.join(root, split, "Images", stem + ".jpg")
            bone_path = os.path.join(root, split, "Bone Level Annotations", stem + ".json")
            mask_dir = os.path.join(root, split, "Masks (Tooth-wise)", stem)
            if not (os.path.exists(img_path) and os.path.isdir(mask_dir)):
                continue
            img = cv2.imread(img_path)
            H, W = img.shape[:2]
            kp = json.load(open(kp_path))
            bone = json.load(open(bone_path)) if os.path.exists(bone_path) else {}
            rows = convert_image(kp, bone, load_masks(mask_dir))
            lines = []
            for r in rows:
                x1, y1, x2, y2 = r["box"]
                vals = [0, (x1 + x2) / 2 / W, (y1 + y2) / 2 / H, (x2 - x1) / W, (y2 - y1) / H]
                for p in (r["cej"], r["apex"], r["crest"]):
                    vals += [p[0] / W, p[1] / H, 2] if p is not None else [0, 0, 0]
                lines.append(" ".join(f"{v:.6f}" if isinstance(v, float) else str(v) for v in vals))
                n_crest += r["crest"] is not None
            if not lines:
                continue
            shutil.copy(img_path, os.path.join(args.out, "images", short, stem + ".jpg"))
            open(os.path.join(args.out, "labels", short, stem + ".txt"), "w").write("\n".join(lines) + "\n")
            n_img += 1
            n_teeth += len(lines)
        stats[short] = {"images": n_img, "teeth": n_teeth, "teeth_with_crest": n_crest}
    open(os.path.join(args.out, "pose.yaml"), "w").write(
        f"path: {os.path.abspath(args.out)}\ntrain: images/train\nval: images/val\ntest: images/test\n"
        "kpt_shape: [3, 3]\nflip_idx: [0, 1, 2]\nnames:\n  0: tooth\n")
    print(json.dumps(stats, indent=2))
    return 0


if __name__ == "__main__":
    raise SystemExit(main())

In [ ]:
# 5. Convert to YOLO-pose (keypoints per tooth: CEJ, root apex, bone crest). Takes a few minutes.
if not os.path.exists("/content/pose/pose.yaml"):
    !python /content/convert_denpar.py --src /content/denpar --out /content/pose
print(open("/content/pose/pose.yaml").read())

In [ ]:
# 6. Train (resumes from Drive after a disconnect)
from ultralytics import YOLO
run_dir = f"{ROOT}/runs/landmarks"
last, done = f"{run_dir}/weights/last.pt", f"{run_dir}/DONE"
if os.path.exists(done):
    print("Landmark model already trained, skipping.")
elif os.path.exists(last):
    YOLO(last).train(resume=True)
else:
    YOLO(f"{MODEL}.pt").train(
        data="/content/pose/pose.yaml", imgsz=IMGSZ, epochs=EPOCHS, patience=PATIENCE, batch=BATCH,
        project=f"{ROOT}/runs", name="landmarks", exist_ok=True, seed=SEED,
        fliplr=0.5, flipud=0.5,            # one tooth class and symmetric keypoints, so flips are safe here
        degrees=5, translate=0.05, scale=0.3, mosaic=1.0, close_mosaic=20,
        hsv_h=0.0, hsv_s=0.0, hsv_v=0.3, cos_lr=True, optimizer="AdamW", lr0=0.001,
        cache="disk", workers=2, plots=True)
open(done, "w").write("ok")

In [ ]:
# 7. Honest evaluation on the TEST split + conformal calibration from the VALIDATION split
import glob, json, math, shutil
import numpy as np
best = f"{ROOT}/runs/landmarks/weights/best.pt"
model = YOLO(best)
pm = model.val(data="/content/pose/pose.yaml", split="test", imgsz=IMGSZ, batch=4,
               project=f"{ROOT}/runs", name="landmarks_test", exist_ok=True)

def bone_loss(cej, apex, crest):          # identical to backend/app/ml/measurement/bone_loss.py
    root = apex - cej; L2 = float(root @ root)
    return None if L2 <= 1e-6 else max(0.0, min(100.0, float((crest - cej) @ root) / L2 * 100))

def stage(p): return "I" if p < 15 else ("II" if p < 33 else "III")

def iou(a, b):
    ix = max(0, min(a[2], b[2]) - max(a[0], b[0])); iy = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    u = (a[2]-a[0])*(a[3]-a[1]) + (b[2]-b[0])*(b[3]-b[1]) - ix*iy
    return ix*iy/u if u else 0

def evaluate(split):
    rows, n_gt, n_found = [], 0, 0
    for img in sorted(glob.glob(f"/content/pose/images/{split}/*.jpg")):
        r = model.predict(img, imgsz=IMGSZ, conf=0.05, verbose=False)[0]
        H, W = r.orig_shape
        boxes = r.boxes.xyxy.cpu().numpy() if r.boxes is not None else np.zeros((0, 4))
        kps = r.keypoints.data.cpu().numpy() if r.keypoints is not None else np.zeros((0, 3, 3))
        used = set()
        for line in open(img.replace("/images/", "/labels/").replace(".jpg", ".txt")):
            v = list(map(float, line.split()))
            cx, cy, w, h = v[1]*W, v[2]*H, v[3]*W, v[4]*H
            gbox = [cx-w/2, cy-h/2, cx+w/2, cy+h/2]
            g = [np.array([v[5+3*k]*W, v[6+3*k]*H]) for k in range(3)]
            gvis = [v[7+3*k] > 0 for k in range(3)]
            n_gt += 1
            cand = [(iou(gbox, b), i) for i, b in enumerate(boxes) if i not in used]
            if not cand or max(cand)[0] < 0.5: continue
            _, i = max(cand); used.add(i); n_found += 1
            p = [kps[i, k, :2] for k in range(3)]; conf = [float(kps[i, k, 2]) for k in range(3)]
            root_len = float(np.linalg.norm(g[1] - g[0]))
            err = {name: float(np.linalg.norm(p[k] - g[k])) / root_len * 100
                   for k, name in enumerate(("cej", "apex", "crest")) if gvis[k]}
            ref = bone_loss(*g) if all(gvis) else None
            pred = bone_loss(*p)
            rows.append({"err_pct_root": err, "ref": ref, "pred": pred, "confident": min(conf) >= MIN_KPT_CONF})
    return rows, n_gt, n_found

def summary(rows, n_gt, n_found):
    bl = [(r["pred"], r["ref"]) for r in rows if r["ref"] is not None and r["pred"] is not None]
    conf_bl = [(r["pred"], r["ref"]) for r in rows if r["ref"] is not None and r["pred"] is not None and r["confident"]]
    def blstats(pairs):
        if not pairs: return None
        e = np.array([abs(p - q) for p, q in pairs])
        return {"n_teeth": len(pairs), "bone_loss_MAE_pct_points": round(float(e.mean()), 2),
                "bone_loss_median_abs_error": round(float(np.median(e)), 2),
                "within_5_points": round(float((e <= 5).mean()), 3), "within_10_points": round(float((e <= 10).mean()), 3),
                "stage_agreement": round(float(np.mean([stage(p) == stage(q) for p, q in pairs])), 3)}
    return {
        "teeth_labelled": n_gt, "teeth_found_iou50": n_found, "tooth_recall": round(n_found / max(n_gt, 1), 3),
        "landmark_error_pct_of_root_length": {k: round(float(np.mean([r["err_pct_root"][k] for r in rows if k in r["err_pct_root"]])), 2)
                                               for k in ("cej", "apex", "crest")},
        "all_found_teeth": blstats(bl),
        "teeth_above_app_confidence": blstats(conf_bl),
        "share_above_app_confidence": round(float(np.mean([r["confident"] for r in rows])) if rows else 0, 3),
    }

test_rows, tg, tf = evaluate("test")
val_rows, vg, vf = evaluate("val")
metrics = {"dataset": "DenPAR periapical (CC BY 4.0), official split", "model": MODEL, "imgsz": IMGSZ,
           "pose_mAP50_test": round(float(pm.pose.map50), 4), "pose_mAP50_95_test": round(float(pm.pose.map), 4),
           "box_mAP50_test": round(float(pm.box.map50), 4),
           "test": summary(test_rows, tg, tf), "val": summary(val_rows, vg, vf)}

# Conformal: scores from VALIDATION teeth that pass the app's confidence gate; coverage measured on TEST.
def pairs(rows): return [(r["pred"], r["ref"]) for r in rows if r["ref"] is not None and r["pred"] is not None and r["confident"]]
vp, tp = pairs(val_rows), pairs(test_rows)
scores = sorted(abs(p - q) for p, q in vp)
def q_at(cov):
    k = math.ceil((len(scores) + 1) * cov)
    return scores[k - 1] if k <= len(scores) else None
levels = {}
for cov in (0.8, 0.9, 0.95):
    q = q_at(cov)
    levels[str(cov)] = {"q_from_half": q,
                        "empirical_coverage_other_half": round(float(np.mean([abs(p - r) <= q for p, r in tp])), 3) if q is not None and tp else None}
edges = np.linspace(0, 100, 6); bins = []
for lo, hi in zip(edges[:-1], edges[1:]):
    sel = [(p, r) for p, r in tp if lo <= p < hi or (hi == 100 and p == 100)]
    if sel: bins.append({"bin": [float(lo), float(hi)], "n": len(sel), "mean_predicted": round(float(np.mean([p for p, _ in sel])), 2),
                         "mean_reference": round(float(np.mean([r for _, r in sel])), 2)})
import datetime as dt
cal = {"created": dt.datetime.now(dt.timezone.utc).isoformat(),
       "source": "DenPAR periapical radiographs, specialist-verified CEJ / apex / bone-level labels (CC BY 4.0)",
       "notes": "Scores from the DenPAR validation split; 'q_from_half' = q from validation, 'empirical_coverage_other_half' = coverage "
                "measured on the DenPAR test split. Periapical radiographs only; not validated on panoramic images.",
       "n_scores": len(scores), "scores": [round(float(s), 4) for s in scores], "levels": levels,
       "mean_absolute_error_pct": round(float(np.mean(scores)), 3) if scores else None, "reliability_bins": bins,
       "image_type": "periapical"}
json.dump(cal, open(f"{ROOT}/export/conformal_calibration.json", "w"), indent=2)
json.dump(metrics, open(f"{ROOT}/export/landmark_test_metrics.json", "w"), indent=2)
shutil.copy(best, f"{ROOT}/export/dental_landmark_yolov8n-pose.pt")
print(json.dumps(metrics, indent=2))
print("conformal levels (q from val, coverage on test):", json.dumps(levels, indent=2))
print("Saved export/dental_landmark_yolov8n-pose.pt, landmark_test_metrics.json, conformal_calibration.json")

## Done
Download `MyDrive/PerioVision/export/` (right-click → Download), extract it, and give it to Claude,
or run `.\run.ps1 install-models -From <folder>`.